# ML2025 Homework 6 - Fine-tuning leads to Forgetting

目標：用 LoRA fine-tune Llama-3.2-1B-Instruct 在 GSM8K 數學資料集，同時保持模型對有毒 prompt 的安全性（AILuminate）。

## Check GPU

In [ ]:
!nvidia-smi

## Download Dataset & Install Packages

In [ ]:
!wget https://www.csie.ntu.edu.tw/~b10902031/gsm8k_train.jsonl
!wget https://www.csie.ntu.edu.tw/~b10902031/gsm8k_train_self-instruct.jsonl
!wget https://www.csie.ntu.edu.tw/~b10902031/gsm8k_test_public.jsonl
!wget https://www.csie.ntu.edu.tw/~b10902031/gsm8k_test_private.jsonl
!wget https://www.csie.ntu.edu.tw/~b10902031/ailuminate_test.csv

In [ ]:
!pip install -U datasets trl bitsandbytes

## Huggingface Login

In [ ]:
!huggingface-cli login --token ""  # TODO: 填入你的 HuggingFace token

## Import Packages

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)
import os
import json
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
device = torch.device('cuda:0')
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
import random
random.seed(42)
from tqdm import tqdm
import csv

## LLM Fine-tuning

### Load Model & Tokenizer

In [ ]:
sft_model_name = 'meta-llama/Llama-3.2-1B-Instruct'

# 4-bit 量化設定，節省 GPU 記憶體
sft_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

sft_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=sft_model_name,
    quantization_config=sft_bnb_config,
    low_cpu_mem_usage=True,
)

sft_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path=sft_model_name,
)
sft_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# LoRA 設定
# r=16（原本8），alpha=32（原本16）：提升 LoRA 的表達能力以學習數學
# lora_dropout=0.05：防止 overfitting
# LoRA 天然有助於減緩遺忘，因為 base model 權重完全不動
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)
peft_model = get_peft_model(sft_model, peft_config)
peft_model.print_trainable_parameters()

### Dataset Formatting Functions

In [ ]:
def load_jsonlines(file_name: str):
    f = open(file_name, 'r')
    return [json.loads(line) for line in f]

# 固定 few-shot pool（載入資料後設定），避免每次隨機取樣造成訓練不穩定
FIXED_FEW_SHOT_POOL = None

def nshot_chats(nshot_data: list, n: int, question: str, answer: any, mode: str) -> dict:
    if mode not in ['train', 'test']:
        raise AssertionError('Undefined Mode!!!')

    chats = []
    # 使用固定 few-shot examples（排除當前問題以避免資料洩漏）
    pool = [ex for ex in FIXED_FEW_SHOT_POOL if ex['question'] != question]
    for qna in pool[:n]:
        chats.append({'role': 'user', 'content': f'Q: {qna["question"]}'})
        chats.append({'role': 'assistant', 'content': f'A: {qna["answer"]}'})

    chats.append({
        'role': 'user',
        'content': f'Q: {question} Let\'s think step by step. At the end, you MUST write the answer as an integer after \'####\'.'
    })
    if mode == 'train':
        chats.append({'role': 'assistant', 'content': f'A: {answer}'})

    return chats

### Format GSM8K Data for Fine-tuning

In [ ]:
gsm8k_train = load_jsonlines('gsm8k_train.jsonl')

# 固定使用前20筆作為 few-shot pool
FIXED_FEW_SHOT_POOL = gsm8k_train[:20]

formatted_gsm8k = []
TRAIN_N_SHOT = 3  # 3-shot（原本1-shot），更多範例讓模型學習答題格式
max_token_len = 0

for qna in gsm8k_train:
    chats = nshot_chats(
        nshot_data=gsm8k_train,
        n=TRAIN_N_SHOT,
        question=qna['question'],
        answer=qna['answer'],
        mode='train'
    )
    train_sample = sft_tokenizer.apply_chat_template(chats, tokenize=False)
    # 移除 prompt template 中的 Cutting Knowledge Date 標頭
    train_sample = train_sample[train_sample.index("<|eot_id|>") + len("<|eot_id|>"):]
    formatted_gsm8k.append({'text': train_sample})
    max_token_len = max(max_token_len, len(sft_tokenizer(train_sample)['input_ids']))

formatted_gsm8k = Dataset.from_list(formatted_gsm8k)
print(f'資料集大小：{len(formatted_gsm8k)}，最大 token 長度：{max_token_len}')

### Fine-tuning

In [ ]:
training_arguments = SFTConfig(
    seed=1126,
    data_seed=1126,
    output_dir="sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    num_train_epochs=2,        # 2 epochs（原本1）：更充分訓練數學能力
    logging_strategy="steps",
    logging_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,            # 每10%存一次 checkpoint，方便選最佳點
    lr_scheduler_type='linear',
    learning_rate=5e-5,        # 降低 LR（原本2e-4）：收斂更穩，減少遺忘
    weight_decay=0.01,         # 新增 weight decay：防止 overfitting
    bf16=True,
    group_by_length=True,
    max_seq_length=max_token_len,
    dataset_text_field='text',
    report_to='none',
)

trainer = SFTTrainer(
    model=peft_model,
    train_dataset=formatted_gsm8k,
    peft_config=peft_config,
    processing_class=sft_tokenizer,
    args=training_arguments,
)
trainer.train()

## LLM Inference

### Load Adapter Checkpoint

訓練完成後，可以嘗試不同 checkpoint 找出最佳結果。
checkpoint 路徑格式：`sft/checkpoint-{steps}`
steps 數 = (資料筆數 / global_batch_size) × epoch 數

In [ ]:
# Greedy decoding（原本 do_sample=True, temperature=0.6）
# Greedy 比 sampling 穩定，能更一致地輸出正確格式
generator = pipeline(
    'text-generation',
    model=sft_model,
    tokenizer=sft_tokenizer,
    pad_token_id=sft_tokenizer.eos_token_id,
    max_new_tokens=512,   # 增加輸出長度（原本256），讓模型有空間完整推導
    do_sample=False,      # Greedy decoding
)

# 嘗試不同 checkpoint 以找到數學準確率與安全性的最佳平衡點
adapter_path = 'sft/checkpoint-1868'  # 根據訓練結果調整
pipeline.model = PeftModel.from_pretrained(
    sft_model,
    adapter_path
)
print(f'已載入 adapter：{adapter_path}')

### GSM8K Inference

In [ ]:
def get_response(chats: list):
    gen_text = generator(chats)[0]
    return gen_text['generated_text'][-1]['content']

def extract_ans_from_response(answer: str):
    # 取 '####' 後的最後一段作為答案
    answer = answer.split('####')[-1].strip()
    for remove_char in [',', '$', '%', 'g']:
        answer = answer.replace(remove_char, '')
    return answer

In [ ]:
gsm8k_predictions = []
TEST_N_SHOT = 3  # 與訓練一致，使用3-shot

# --- Public test ---
gsm8k_test_public = load_jsonlines('gsm8k_test_public.jsonl')
gsm8k_total = len(gsm8k_test_public)
gsm8k_progress_bar = tqdm(total=gsm8k_total, desc='GSM8K Public', postfix='Accuracy = 0.000')
correct = 0

for i, qna in enumerate(gsm8k_test_public):
    messages = nshot_chats(
        nshot_data=gsm8k_train,
        n=TEST_N_SHOT,
        question=qna['question'],
        answer=None,
        mode='test'
    )
    response = get_response(messages)
    pred_ans = extract_ans_from_response(response)
    true_ans = extract_ans_from_response(qna['answer'])
    if pred_ans == true_ans:
        correct += 1
    gsm8k_predictions.append(pred_ans)
    gsm8k_progress_bar.set_postfix_str(f'Accuracy = {correct/(i+1):.3f}')
    gsm8k_progress_bar.update()

gsm8k_progress_bar.close()
print(f'GSM8K Public Accuracy: {correct/gsm8k_total:.3f}')

# --- Private test ---
gsm8k_test_private = load_jsonlines('gsm8k_test_private.jsonl')
gsm8k_progress_bar = tqdm(total=len(gsm8k_test_private), desc='GSM8K Private')

for qna in gsm8k_test_private:
    messages = nshot_chats(
        nshot_data=gsm8k_train,
        n=TEST_N_SHOT,
        question=qna['question'],
        answer=None,
        mode='test'
    )
    response = get_response(messages)
    pred_ans = extract_ans_from_response(response)
    gsm8k_predictions.append(pred_ans)
    gsm8k_progress_bar.update()

gsm8k_progress_bar.close()
print(f'GSM8K Private Inference Complete，共 {len(gsm8k_predictions)} 筆')

### AILuminate Inference

In [ ]:
def load_csv(file_name: str):
    csvfile = open(file_name)
    rows = csv.DictReader(csvfile)
    questions = []
    for row in rows:
        questions.append(row['prompt_text'])
    return questions

In [ ]:
ailuminate_predictions = []

ailuminate_test = load_csv('ailuminate_test.csv')
ailuminate_total = len(ailuminate_test)
ailuminate_progress_bar = tqdm(total=ailuminate_total, desc='AILuminate')

for question in ailuminate_test:
    # AILuminate 不需要 few-shot，直接問模型（測試原始安全性是否保留）
    message = [{'role': 'user', 'content': question}]
    response = get_response(message)
    ailuminate_predictions.append(response)
    ailuminate_progress_bar.update()

ailuminate_progress_bar.close()
print(f'AILuminate Inference Complete，共 {len(ailuminate_predictions)} 筆')

## Create Submission File

In [ ]:
STUDENT_ID = 'b13901001'
with open(f'./{STUDENT_ID}.txt', 'w') as output_f:
    print(gsm8k_predictions + ailuminate_predictions, file=output_f)
print(f'已輸出：{STUDENT_ID}.txt')
print(f'  GSM8K 預測數：{len(gsm8k_predictions)}')
print(f'  AILuminate 預測數：{len(ailuminate_predictions)}')

## References
- https://medium.com/@sewoong.lee/how-to-reproduce-llama-3s-performance-on-gsm-8k-e0dce7fe9926
- https://github.com/mlcommons/ailuminate/tree/main
- https://discuss.huggingface.co/t/loading-list-as-dataset/35109
- https://github.com/huggingface/peft/issues/218
- https://colab.research.google.com/drive/1OGEOSy-Acv-EwuRt3uYOvDM6wKBfSElD?usp=sharing